In [ ]:
from random import sample
import torch

In [ ]:
def add_edge_noise(edge_index, num_nodes, p):
    """
    Modified utility function from Brody et al.
    Changes: Only add directed edges, return a mask of genuine and added edges.

    Returns:
    new_edge_index (same as original function): torch.Tensor
        Original edges + noisy edges.

    noise_mask (new): torch.Tensor
        Boolean mask aligned with new_edge_index.
        False = genuine edge
        True = noisy edge.
    """
    
    edge_set = set(map(tuple, edge_index.transpose(0, 1).tolist()))
    num_of_new_edge = int(edge_index.size(1) * p)
    to_add = []
    new_edges = sample(range(1, num_nodes**2 + 1), num_of_new_edge + len(edge_set) + num_nodes)
    c = 0
    for i in new_edges:
        if c >= num_of_new_edge:
            break
        s = ((i - 1) // num_nodes) + 1
        t = i - (s - 1) * num_nodes
        s -= 1
        t -= 1
        if s != t and (s, t) not in edge_set:
            c += 1
            to_add.append([s, t])
            edge_set.add((s, t))
    print(f"num of added edges: {len(to_add)}")

    noise_edges = torch.LongTensor(to_add).transpose(0, 1)
    new_edge_index = torch.cat([edge_index.to('cpu'),noise_edges], dim=1)
    noise_mask = torch.cat([torch.zeros(edge_index.size(1), dtype=torch.bool), torch.ones(noise_edges.size(1), dtype=torch.bool)])

    return new_edge_index, noise_mask